## 01 — What We Built

Before moving on to the library version, we take stock.

Good engineers do not just ship and move on — they understand what they built, where it breaks, and what it would cost to fix those things. This notebook measures the handbuilt system honestly.

## What We Built — Component Inventory

| Component | Where | What it does |
|-----------|-------|--------------|
| **Douglas-Peucker** | Module 01 | Reduces point count per line segment |
| **LOD pipeline** | Module 02 | Produces 4 simplified GeoJSON files |
| **Bbox computation** | Module 03 | Gets the extent of any feature |
| **Bbox intersection** | Module 03 | Tests if a feature overlaps the viewport |
| **Uniform grid index** | Module 04 | Buckets features for fast viewport queries |
| **LOD decision function** | Module 05 | Selects the right file by zoom level |
| **Live map viewer** | Module 06 | Wires everything into an interactive display |

Each component was built from scratch. We understand every line.

## Measuring the System

In [1]:
import json
import time
from pathlib import Path

lod_dir  = Path("../../data/lod")
raw_path = Path("../../data/ne_10m_railroads.geojson")

lod_files = {
    "coarse":     "railroads_coarse.geojson",
    "medium":     "railroads_medium.geojson",
    "fine":       "railroads_fine.geojson",
    "extra_fine": "railroads_extra_fine.geojson",
}

print(f"{'File':<28} {'Size (MB)':>10} {'Features':>10} {'Total pts':>12} {'Load (s)':>10}")
print("-" * 75)

for label, filename in [("original", None)] + list(lod_files.items()):
    path = raw_path if filename is None else lod_dir / filename
    t0 = time.perf_counter()
    with open(path) as f:
        data = json.load(f)
    load_time = time.perf_counter() - t0
    feats = data["features"]
    n_pts = sum(len(f["geometry"]["coordinates"]) for f in feats)
    size  = path.stat().st_size / 1_000_000
    print(f"{label:<28} {size:>10.2f} {len(feats):>10,} {n_pts:>12,} {load_time:>10.3f}")

File                          Size (MB)   Features    Total pts   Load (s)
---------------------------------------------------------------------------
original                          39.60     25,413    1,396,480      0.669
coarse                             1.08      2,845        5,690      0.012
medium                             9.66     25,413       55,516      0.103
fine                              11.34     25,413      124,844      0.127
extra_fine                        18.98     25,413      441,890      0.236


## Where the System Still Hurts

The viewer works. But it has real limitations. Let's name them honestly.

### Pain Point 1 — Startup Cost

Every session, we load 4 files and build 4 grid indexes. This takes several seconds before the map is usable.

A tile server has no startup cost — tiles are pre-built and stored. The server just reads a file from a database and sends it.

In [2]:
def feature_bbox(feature):
    coords = feature["geometry"]["coordinates"]
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return [min(lons), min(lats), max(lons), max(lats)]

class GridIndex:
    CELL_SIZE = 10.0
    def __init__(self): self.cells = {}
    def _cells(self, bbox):
        lo, la, hi, ha = bbox; cs = self.CELL_SIZE
        return [(c, r) for c in range(int((lo+180)/cs), int((hi+180)/cs)+1)
                       for r in range(int((la+ 90)/cs), int((ha+ 90)/cs)+1)]
    def build(self, features):
        self.cells = {}
        for i, f in enumerate(features):
            for cell in self._cells(feature_bbox(f)): self.cells.setdefault(cell,[]).append((i,f))

total_startup = 0
for filename in lod_files.values():
    t0 = time.perf_counter()
    with open(lod_dir / filename) as f:
        feats = json.load(f)["features"]
    idx = GridIndex()
    idx.build(feats)
    elapsed = time.perf_counter() - t0
    total_startup += elapsed

print(f"Total startup time (load + index build): {total_startup:.2f}s")

Total startup time (load + index build): 0.71s


### Pain Point 2 — GeoJSON Is Verbose

GeoJSON is human-readable text. Every coordinate is stored as a decimal number string. A production mapping pipeline uses **binary encoding** (Mapbox Vector Tiles, MVT) which stores coordinates as integers relative to the tile origin — 5–10× smaller than equivalent GeoJSON and much faster to parse.

In [3]:
# Rough estimate: how large would our files be in a binary format?
# MVT stores coordinates as 2-byte integers per axis
# GeoJSON stores them as ~8-character float strings

GEOJSON_BYTES_PER_COORD = 16   # avg chars for [lon, lat] pair including punctuation
MVT_BYTES_PER_COORD     = 4    # 2 bytes each for x, y as zigzag-encoded varint

for filename in lod_files.values():
    with open(lod_dir / filename) as f:
        feats = json.load(f)["features"]
    total_pts = sum(len(f["geometry"]["coordinates"]) for f in feats)
    actual_mb  = (lod_dir / filename).stat().st_size / 1_000_000
    est_mvt_mb = total_pts * MVT_BYTES_PER_COORD / 1_000_000
    print(f"{filename:<38} actual: {actual_mb:.2f}MB   est MVT: {est_mvt_mb:.2f}MB")

railroads_coarse.geojson               actual: 1.08MB   est MVT: 0.02MB
railroads_medium.geojson               actual: 9.66MB   est MVT: 0.22MB
railroads_fine.geojson                 actual: 11.34MB   est MVT: 0.50MB
railroads_extra_fine.geojson           actual: 18.98MB   est MVT: 1.77MB


### Pain Point 3 — The Whole File Is Always Resident

To query the fine LOD for Paris, we load the entire `railroads_fine.geojson` into memory — including Australia, South America, and Russia. A tile system would read only the Paris tile from a database, never touching the rest.

Our grid index helps at query time, but the full file still had to load first.

### Pain Point 4 — No Partial Load or Streaming

When the user pans to a new region, we re-query the index immediately — but the data was all loaded at startup. A tile server streams only the tiles the user actually views. If the user never visits Australia, those tiles are never fetched.

## The Decision Inventory

Every system embeds design decisions. Here are ours, stated explicitly:

| Decision | What we chose | What we gave up |
|----------|--------------|------------------|
| File format | GeoJSON (text) | Binary efficiency |
| Simplification algorithm | Douglas-Peucker via Shapely | Topology-preserving alternatives |
| LOD levels | 4 fixed levels | Continuous zoom-adaptive detail |
| Coarse filter | scalerank ≤ 4 | Coverage in scalerank 5+ regions |
| Spatial index | Uniform 10° grid | Adaptive indexes (R-tree, quadtree) |
| Culling granularity | Feature bbox | True geometry intersection |
| Transition policy | Fixed zoom thresholds | Hysteresis (implemented but not used in final viewer) |
| Memory model | All data loaded at startup | Lazy / tile-based loading |

None of these decisions are wrong. They are appropriate for a teaching system built from scratch. A production system makes different choices for different reasons.

## Exercise A

Measure the peak memory usage of the viewer at startup (after all 4 LOD files are loaded and all 4 indexes are built).

Use Python's `tracemalloc` module:

```python
import tracemalloc
tracemalloc.start()
# ... load and build ...
current, peak = tracemalloc.get_traced_memory()
print(f"Peak memory: {peak / 1_000_000:.1f} MB")
```

How does this compare to just reading the four files without building indexes?

In [4]:
# Measure peak memory: (a) loading files only, (b) loading + building indexes
# Your code here

import tracemalloc

# (a) Loading files only
tracemalloc.start()

lod_data = {}

for filename in lod_files.values():
    with open(lod_dir / filename) as f:
        lod_data[filename] = json.load(f)

current, peak_load_only = tracemalloc.get_traced_memory()

tracemalloc.stop()

print(f"Peak memory (loading only): {peak_load_only / 1_000_000:.1f} MB")


# (b) Loading + building indexes
tracemalloc.start()

lod_data = {}
indexes = []

for filename in lod_files.values():

    with open(lod_dir / filename) as f:
        data = json.load(f)

    lod_data[filename] = data

    idx = GridIndex()
    idx.build(data["features"])

    indexes.append(idx)

current, peak_with_indexes = tracemalloc.get_traced_memory()

tracemalloc.stop()

print(f"Peak memory (loading + indexes): {peak_with_indexes / 1_000_000:.1f} MB")

Peak memory (loading only): 208.5 MB
Peak memory (loading + indexes): 214.1 MB


Peak memory usage for loading the four GeoJSON files only was 208.5 MB. After building the grid indexes, peak memory usage increased to 214.1 MB. The increase comes from storing additional feature references inside the spatial index cells.

## Exercise B

Write a short summary (8–12 sentences) of the Railroad LOD system as if you were presenting it to a team that had never seen it.

Cover:
- What problem it solves
- What the four major components are
- What the main performance tradeoffs are
- What you would change if this needed to serve 10 million users instead of one notebook

Write it in the cell below as markdown.

# Railroad LOD System Summary

The Railroad LOD system is designed to display a very large railroad dataset efficiently on an interactive web map. The main problem it solves is that rendering the full dataset at every zoom level would be too slow and would overload the browser with unnecessary geometry detail.

The system uses four major components. First, the Douglas-Peucker simplification algorithm reduces the number of points in railroad geometries. Second, the LOD pipeline creates four GeoJSON files with different levels of detail for different zoom ranges. Third, a uniform grid spatial index allows fast viewport queries by dividing the world into cells and storing feature references inside them. Fourth, the live viewer dynamically switches between LOD levels and queries the correct index as the user pans and zooms.

The main performance tradeoff is between simplicity and scalability. GeoJSON is easy to understand and debug, but it is verbose and memory-intensive compared to binary vector tile formats. The system also loads all files and indexes into memory at startup, which increases startup time and memory usage. The uniform grid index is simple and fast, but it does not adapt well to uneven feature density like an R-tree would.

If this system needed to support 10 million users instead of a single notebook, I would replace GeoJSON with vector tiles such as Mapbox Vector Tiles. I would move processing to a server-side tile pipeline with streaming and lazy loading so clients only download visible regions. I would also replace the uniform grid with a production spatial index such as an R-tree or STRtree and cache frequently requested tiles to reduce server load.

## Check Your Understanding

We identified four pain points: startup cost, verbose format, whole-file loading, and no streaming.

Rank them from **most impactful** to **least impactful** for a user on a slow connection (e.g., mobile data). Justify your ranking in 2–3 sentences.

---

# Most impactful: verbose GeoJSON format.
 Large GeoJSON files are expensive to download on a slow mobile connection and significantly increase loading time.

# Second: whole-file loading.
 The client downloads and keeps the entire dataset in memory even if the user only views a small region.

# Third: no streaming.
#Without streaming, unnecessary data is transferred up front instead of loading incrementally as the user pans.

# Least impactful: startup cost.
 Startup processing time matters, but network transfer size is usually the largest bottleneck on slow mobile connections.

## Next

In [Module 07 — The Library Version](../07-The_Library_Version/README.md), we hand the problem to `tippecanoe` and see how a professional tool addresses every one of these pain points.